Notebook References:

- xarray access: https://github.com/EOPF-Sample-Service/eopf-sample-notebooks/blob/main/notebooks/Sentinel-2/Sentinel-2_L1C_MSI_Zarr_product_exploration.ipynb
- xcube-eopf: https://eopf-sample-service.github.io/eopf-sample-notebooks/introduction-xcube-eopf-plugin
- xcube-stac: https://github.com/xcube-dev/xcube-stac/blob/main/examples/notebooks/cdse_sentinel_2.ipynb

`conda install xcube-stac xcube-eopf`
`conda install -c conda-forge libgdal-jp2openjpeg`


In [1]:
import cartopy.crs as ccrs
import numpy as np
import xarray as xr
import requests

import dask
from xcube.core.store import new_data_store, get_data_store_params_schema
from xcube_eopf.utils import reproject_bbox

#for direct loading
import rioxarray
import fsspec
import s3fs

# for benchmarking
from dataclasses import dataclass
from typing import List
from itertools import product
import pandas as pd
import time

### SAFE on CDSE S3

In [8]:
# cdse credentials
credentials = {
    "key": "FTE4ZT820RDZTHOU6I8C",
    "secret": "EdSaK2k1DjJm1rTlbucDaaSsmSSawWFz9da9Wemz",
}

### Functions

In [10]:
def create_aoi(bbox, reduction):
    """
    Generate a reduced bounding box or centroid based on a reduction factor.
    Helper function to easily create portions of the original bbox around the centroid.

    Parameters:
    - bbox: [min_lon, min_lat, max_lon, max_lat]
    - reduction: float between 0 and 1
        - 0 returns the centroid as (lon, lat)
        - 0 < reduction < 1 returns a scaled bounding box centered at the centroid

    Returns:
    - reduced bounding box list
    """
    if not (0 <= reduction <= 1):
        raise ValueError("Reduction must be between 0 and 1.")

    min_lon, min_lat, max_lon, max_lat = bbox

    # Compute centroid
    centroid_lon = (min_lon + max_lon) / 2
    centroid_lat = (min_lat + max_lat) / 2

    #if reduction == 0:
     #   return (centroid_lon, centroid_lat)

    # Compute reduced bounding box dimensions
    lat_span = (max_lat - min_lat) * reduction
    lon_span = (max_lon - min_lon) * reduction

    return [
        centroid_lon - lon_span / 2,
        centroid_lat - lat_span / 2,
        centroid_lon + lon_span / 2,
        centroid_lat + lat_span / 2,
    ]

In [11]:
# this is used for defining inputs 
@dataclass
class BenchmarkConfig:
    data_id: str
    bbox: List[float]
    time_range: List[str]
    spatial_res: int
    crs: str
    variables: List[str] 


In [12]:
# access function eopf
def access_eopf(cfg: BenchmarkConfig):
    return store_zarr.open_data(
        data_id=cfg.data_id,
        bbox=reproject_bbox(cfg.bbox, "EPSG:4326", cfg.crs), # has to be done for xcube # TODO: throws error with dfg.crs = EPSG:4326
        time_range=cfg.time_range,
        spatial_res=cfg.spatial_res,
        crs=cfg.crs,
        variables=cfg.variables,
    ).load()


In [13]:
# access function safe
def access_safe(cfg: BenchmarkConfig):
    return store_safe.open_data(
        data_id=cfg.data_id,
        bbox=reproject_bbox(cfg.bbox, "EPSG:4326", cfg.crs), # has to be done for xcube # TODO: throws error with dfg.crs = EPSG:4326
        time_range=cfg.time_range,
        spatial_res=cfg.spatial_res,
        crs=cfg.crs,
        asset_names=[v.upper() for v in cfg.variables],
    ).load()


In [14]:
# benchmark function
# loops through the given configs
def benchmark_data_access(configs, access_fn):
    results = []
    
    for cfg in configs:
        print(f"Running: {cfg}")
        start = time.perf_counter()
        ds = access_fn(cfg)
        end = time.perf_counter()

        n_pixels_xy = ds.sizes['x'] * ds.sizes['y'] # get pixel count
        results.append({
            "data_id": cfg.data_id,
            "bbox": cfg.bbox,
            "time_range": cfg.time_range,
            "spat_res": cfg.spatial_res,
            "crs": cfg.crs,
            "variables": cfg.variables,
            "n_pixels_xy": n_pixels_xy,
            "duration_sec": round(end - start, 4)
        })
        
    return pd.DataFrame(results)


In [ ]:
# pull the bbox from the catalog/object here
url = "https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316"
response = requests.get(url)
item = response.json()
bbox = item["bbox"]
print(bbox)

In [ ]:
# pull native crs
crs_native = item['properties']['proj:code'] # "EPSG:32632"
print(crs_native)

In [ ]:
# define data id
opt_data_id = [
    "sentinel-2-l2a"
]

In [53]:
# define bboxes
# only in lat/lon, reprojection of bbox to chosen crs happens later in code
opt_bbox = [
    #create_aoi(bbox, 0), # pixel TODO: ERROR
    create_aoi(bbox, 256 / 10980), # ml patch approx 256*256
    #create_aoi(bbox, 0.125), # eight
    create_aoi(bbox, 0.25), # quarter
    #bbox, # full  
]

In [44]:
# define crs
# mandatory in xcube
# if it differs from native crs processing is enforced (reprojection, resampling)
opt_crs = [
    crs_native, 
    #"EPSG:4326", # # TODO: reproject_bbox in access_ throws error with dfg.crs = EPSG:4326
    #"EPSG:3035",
]

In [63]:
# define times
opt_time_range = [
    ["2025-05-01", "2025-06-01"], # day
    ["2025-05-01", "2025-05-07"], # month
    #["2024-05-01", "2025-05-01"] # year
]

In [46]:
# define spatial resolution
# everything deviating from native resolution enforces processing (resampling)
opt_spatial_res = [
    10, 
    #20, 
    100,
]

In [47]:
# define band combinations
# choosing bands with different resolutions enforces processing (resampling)
opt_variables = [
    ["b02"],
    ["b02", "b04"],
    #[]
]


## 3. Custom Configurations

### EOPF

In [74]:
# Create custom dataclass object
custom_cfg = BenchmarkConfig(
    data_id=opt_data_id[0],
    bbox=opt_bbox[0],
    time_range=opt_time_range[0],
    spatial_res=opt_spatial_res[0],
    crs=opt_crs[0],
    variables=opt_variables[0]
)
print(custom_cfg)

BenchmarkConfig(data_id='sentinel-2-l2a', bbox=[9.669372670305636, 53.64026948239441, 9.701345402674315, 53.66341039786631], time_range=['2025-05-01', '2025-06-01'], spatial_res=10, crs='EPSG:32632', variables=['b02'])


In [75]:
%%time

bbox_repr = reproject_bbox(custom_cfg.bbox, "EPSG:4326", custom_cfg.crs)
print(bbox_repr)

ds_zarr = store_zarr.open_data(
    data_id=custom_cfg.data_id,
    bbox=bbox_repr,
    time_range=custom_cfg.time_range,
    spatial_res=custom_cfg.spatial_res,
    crs=custom_cfg.crs,
    variables=custom_cfg.variables,
)
ds_zarr.load()

(544229.9589484755, 5943707.259587298, 546367.9651581453, 5946302.084300316)
CPU times: user 3.51 s, sys: 197 ms, total: 3.71 s
Wall time: 11.3 s


<xarray.Dataset> Size: 6MB
Dimensions:      (time: 13, y: 261, x: 215)
Coordinates:
  * time         (time) datetime64[ns] 104B 2025-05-01T10:40:41.025000 ... 20...
    spatial_ref  int64 8B 0
  * x            (x) float64 2kB 5.442e+05 5.442e+05 ... 5.464e+05 5.464e+05
  * y            (y) float64 2kB 5.946e+06 5.946e+06 ... 5.944e+06 5.944e+06
Data variables:
    b02          (time, y, x) float64 6MB nan nan nan nan ... nan nan nan nan
Attributes: (4)

### SAFE

In [84]:
# bands have to be renamed
bands_cdse = ['B02']

In [85]:
%%time
ds_safe = store_safe.open_data(
    data_id=custom_cfg.data_id,
    bbox=bbox_repr,
    time_range=custom_cfg.time_range,
    spatial_res=custom_cfg.spatial_res,
    crs=custom_cfg.crs,
    asset_names=bands_cdse,
)
ds_safe.load()

CPU times: user 6.06 s, sys: 771 ms, total: 6.83 s
Wall time: 1min 24s


<xarray.Dataset> Size: 2MB
Dimensions:      (time: 16, y: 260, x: 214)
Coordinates:
  * time         (time) datetime64[ns] 128B 2025-05-01T10:40:41.025000 ... 20...
  * x            (x) float64 2kB 5.442e+05 5.442e+05 ... 5.464e+05 5.464e+05
  * y            (y) float64 2kB 5.946e+06 5.946e+06 ... 5.944e+06 5.944e+06
    spatial_ref  int64 8B 0
Data variables:
    B02          (time, y, x) uint16 2MB 1540 1619 1425 1324 ... 6412 6508 6700
Attributes: (13)